In [ ]:
import random # 무작위 숫자 추출 모듈.
import collections # BFS (너비 우선 탐색) 큐(deque) 사용.
import math # 수학 계산, 특히 `floor` 함수 필요.

# 게임 설정.
BOARD_SIZE = 9 # 지뢰찾기 판 크기, 9x9 칸.
NUM_MINES = 10 # 총 지뢰 개수 10개.

# 게임 상태 저장 변수들.
array = [] # 각 칸 정보 저장. 지뢰(100), 주변 지뢰 개수(0~8).
mine = [] # 실제 지뢰 위치 표시 (1: 지뢰, 0: 비지뢰).
not_mine = [] # 지뢰 아닌 칸 표시 (1: 닫힘, None: 열림). 승리 판단용.
flag_qty = 0 # 남은 깃발 개수 확인 변수.
game = True # 게임 진행 상태 (True: 진행 중, False: 종료).
time_elapsed = 0 # 게임 시작 후 경과 시간 (텍스트 게임에서 미사용).
opened_arr = [] # 칸 열림 상태 기록 (1: 열림, None: 닫힘).
flagged_arr = [] # 칸 깃발 표시 상태 기록 (1: 깃발, None: 무깃발).

# 게임 시작. 보드 초기화 및 지뢰 심기 함수.
def reset():
    global array, mine, not_mine, flag_qty, opened_arr, flagged_arr

    # 모든 변수 초기 상태로 재설정.
    array = []
    mine = []
    not_mine = []
    opened_arr = []
    flagged_arr = []

    # 9x9 보드 생성 과정.
    for i in range(BOARD_SIZE):
        cells = [] # `array` 행 생성 변수.
        cells2 = [] # `not_mine` 행 생성 변수.
        for j in range(BOARD_SIZE):
            cells.append(0) # 칸 주변 지뢰 개수 0으로 시작.
            cells2.append(1) # 모든 칸 '지뢰 아님' 상태(닫힘) 설정.
        array.append(cells)
        not_mine.append(cells2)
        mine.append([0] * BOARD_SIZE) # `mine` 배열 0으로 초기화 (지뢰 없음).

    # `opened_arr`, `flagged_arr` 초기화. 모든 칸 닫힘, 깃발 없음.
    opened_arr = [[None for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]
    flagged_arr = [[None for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

    # 지뢰 배치 시 주변 칸 숫자 증가 보조 함수.
    # (k, l) 위치에 지뢰 생성 시, 주변 8칸의 지뢰 카운트 1씩 증가.
    def add_num(k, l, a):
        if l == 0: # 왼쪽 끝 열(l=0)에 지뢰 존재 시.
            for b in range(2):
                array[k + a][l + b] += 1
        elif l == BOARD_SIZE - 1: # 오른쪽 끝 열(l=BOARD_SIZE - 1)에 지뢰 존재 시.
            for b in range(-1, 1):
                array[k + a][l + b] += 1
        else: # 일반적인 중간 열에 지뢰 존재 시.
            for b in range(-1, 2):
                array[k + a][l + b] += 1

    flag_qty = 0 # 깃발 개수 0으로 재시작 (지뢰 배치 시 증가).
    q_count = 0 # 배치된 지뢰 개수 카운터.
    while q_count < NUM_MINES: # 10개 지뢰 배치 완료까지 반복.
        k = math.floor(random.random() * BOARD_SIZE) # 0-8 범위 랜덤 행(x좌표) 선택.
        l = math.floor(random.random() * BOARD_SIZE) # 0-8 범위 랜덤 열(y좌표) 선택.

        if array[k][l] < 100: # 선택 칸에 지뢰 미존재 시 (100 미만).
            array[k][l] = 100 # 해당 칸 지뢰 표시 (100).
            mine[k][l] = 1 # `mine` 배열에 지뢰 위치 1로 표시.
            not_mine[k][l] = None # 지뢰 칸은 '지뢰 아닌 칸' 목록에서 제외.
            flag_qty += 1 # 깃발 개수 1 증가 (지뢰 위치 추정용).

            # 지뢰 배치 칸 주변 숫자 업데이트.
            if k == 0: # 맨 위 행(k=0)에 지뢰 존재 시.
                for a in range(2):
                    add_num(k, l, a)
            elif k == BOARD_SIZE - 1: # 맨 아래 행(k=BOARD_SIZE - 1)에 지뢰 존재 시.
                for a in range(-1, 1):
                    add_num(k, l, a)
            else: # 일반적인 중간 행에 지뢰 존재 시.
                for a in range(-1, 2):
                    add_num(k, l ,a)
            q_count += 1 # 지뢰 배치 성공 시 카운터 증가.
        # else: 이미 지뢰 있는 칸 선택 시, 카운터 증가 없이 재탐색.

# 게임 보드 화면 출력 함수.
def display_board():
    # 열(가로) 번호 출력.
    print('\n  ' + ' '.join(str(i) for i in range(BOARD_SIZE)))
    print('  ' + '--' * BOARD_SIZE) # 구분선 추가.

    # 각 행(세로) 순회 및 칸 상태 출력.
    for r in range(BOARD_SIZE):
        row_str = f'{r}|' # 행 번호 먼저 표시.
        for c in range(BOARD_SIZE):
            if flagged_arr[r][c]: # 깃발 꽂힌 칸.
                row_str += ' F' # 'F'로 표시.
            elif opened_arr[r][c]: # 칸이 열린 상태.
                if mine[r][c] == 1: # 열린 칸이 지뢰.
                    row_str += ' *' # '*'로 표시.
                else: # 지뢰 아닌 숫자 칸.
                    row_str += f' {array[r][c]}' # 해당 숫자 표시.
            else: # 칸이 닫힌 상태.
                row_str += ' #' # '#'로 표시.
        print(row_str)
    print(f'Flags remaining: {flag_qty}') # 남은 깃발 개수 출력.

# 게임 진행 상태 설정 함수.
def game_set(is_game_running):
    global game
    game = is_game_running # `game` 변수 `True` 또는 `False` 업데이트.

# 깃발 꽂기/뽑기 함수.
def flag(x, y):
    global flag_qty
    if game: # 게임 진행 중 작동.
        if not opened_arr[x][y]: # 미개방 칸에만 깃발 조작 가능.
            if not flagged_arr[x][y]: # 깃발 미존재 칸.
                flagged_arr[x][y] = 1 # 깃발 표시 상태 변경.
                flag_qty -= 1 # 깃발 개수 1 감소.
            else: # 깃발 존재 칸.
                flagged_arr[x][y] = None # 깃발 제거 상태 변경.
                flag_qty += 1 # 깃발 개수 1 증가.

# 특정 칸 열기 함수 (마우스 클릭과 유사).
def click(x, y):
    # (x, y) 좌표 보드 범위(0-BOARD_SIZE-1) 확인.
    if BOARD_SIZE > x >= 0 and BOARD_SIZE > y >= 0:
        if game: # 게임 진행 중 작동.
            # 칸 유효성(None 아님), 미개방, 깃발 미표시 시에만 클릭 처리.
            if (array[x][y] is not None) and (not opened_arr[x][y]) and (not flagged_arr[x][y]):
                opened_arr[x][y] = 1 # 칸 열림 상태로 변경.

# BFS (너비 우선 탐색) 큐(Queue) 자료구조 클래스.
class Queue:
    def __init__(self):
        self.items = collections.deque() # `collections.deque`는 리스트 대비 빠른 앞뒤 추가/제거.

    def enqueue(self, element):
        self.items.append(element) # 큐 맨 뒤 데이터 추가.

    def dequeue(self):
        return self.items.popleft() # 큐 맨 앞 데이터 추출.

    def is_empty(self):
        return len(self.items) == 0 # 큐 비어있음 여부 확인.

# BFS용 주변 8칸 이동 (델타 값).
# `dx[i]`, `dy[i]` 쌍은 현재 칸에서 주변 8칸으로 이동 시 (x, y) 좌표 변화.
dx = [-1, -1, -1, 0, 0, 1, 1, 1] # x (행) 방향 변화량.
dy = [-1, 0, 1, -1, 1, -1, 0, 1] # y (열) 방향 변화량.

# BFS (너비 우선 탐색) 함수. 0(빈 칸) 클릭 시 주변 빈 칸 자동 개방.
def bfs(x, y):
    q = Queue() # 큐 생성.
    q.enqueue([x, y]) # 시작 칸 큐에 추가.

    # 시작 칸 열림 상태로 표시.
    if (array[x][y] is not None) and (not opened_arr[x][y]) and (not flagged_arr[x][y]):
        opened_arr[x][y] = 1

    while not q.is_empty(): # 큐가 비어있지 않은 동안 반복.
        current_x, current_y = q.dequeue() # 큐에서 가장 오래된 칸 추출.

        for i in range(8): # 주변 8칸 탐색.
            new_x = current_x + dx[i] # 새로운 x 좌표 계산.
            new_y = current_y + dy[i] # 새로운 y 좌표 계산.

            # 새로운 좌표 보드 범위 내 존재 여부 확인.
            if BOARD_SIZE > new_x >= 0 and BOARD_SIZE > new_y >= 0:
                # 게임 중, 미개방, 깃발 미표시 시에만 처리.
                if game and (not opened_arr[new_x][new_y]) and (not flagged_arr[new_x][new_y]):
                    if array[new_x][new_y] == 0: # 주변 칸 0(빈 칸).
                        q.enqueue([new_x, new_y]) # 큐에 추가, 주변 탐색 진행.
                        opened_arr[new_x][new_y] = 1 # 해당 칸 열림 상태로 변경.
                    else: # 주변 칸 0 아닌 숫자 칸 (비지뢰).
                        opened_arr[new_x][new_y] = 1 # 해당 칸 열림 상태로 변경 (추가 탐색 없음).

# 게임 승리 조건 확인 함수.
def check_win_condition():
    # 모든 지뢰 아닌 칸 개방 여부 확인.
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if not_mine[i][j] is not None: # 해당 칸 지뢰 아님.
                if opened_arr[i][j] != 1: # 아직 미개방.
                    return False # 승리 조건 미충족.
    return True # 모든 지뢰 아닌 칸 개방 완료, 승리.

# 삽으로 칸 파헤치기(개방) 함수.
def shovel(x, y):
    global game
    if game: # 게임 진행 중 작동.
        if not opened_arr[x][y] and not flagged_arr[x][y]: # 이미 열렸거나 깃발 꽂힌 칸 조작 불가.
            if not mine[x][y]: # 파헤친 곳 지뢰 아님.
                click(x, y) # 해당 칸 개방.
                if check_win_condition(): # 칸 개방 후 승리 조건 확인.
                    game_set(False) # 승리 시 게임 종료.
                    print('\nGAME CLEAR! You won!') # 승리 메시지 출력.
                elif array[x][y] == 0: # 파헤친 칸 0(주변 지뢰 없는 빈 칸).
                    bfs(x, y) # BFS로 주변 빈 칸 및 숫자 칸 자동 개방.
            else: # 파헤친 곳 지뢰.
                game_set(False) # 게임 종료.
                print('\nGAME OVER! You hit a mine!') # 게임 오버 메시지 출력.
                # 게임 오버 시 모든 지뢰 위치 표시.
                for i in range(BOARD_SIZE):
                    for j in range(BOARD_SIZE):
                        if mine[i][j] == 1: # 지뢰 위치 칸.
                            opened_arr[i][j] = 1 # 열림 상태로 표시, 지뢰 노출.

# 텍스트 기반 지뢰찾기 게임 메인 함수.
def play_minesweeper():
    global game, time_elapsed
    reset() # 게임 초기화 (보드, 지뢰 배치 등).
    game_set(True) # 게임 상태 '진행 중' 설정.
    time_elapsed = 0 # 게임 시간 0으로 초기화.

    print("Welcome to Text-Based Minesweeper!")
    print("명령어는 'action row col' 형태로 입력.")
    print("명령어 종류: 's'는 삽으로 파헤치기, 'f'는 깃발 꽂기/뽑기.")
    print("예시: 's 0 0'은 (0, 0) 칸 파헤치기, 'f 1 2'는 (1, 2) 칸에 깃발 꽂기/뽑기.\n")

    while game: # 게임 진행 중 반복.
        display_board() # 현재 보드 상태 화면 출력.
        try:
            user_input = input("당신의 움직임: ").strip().lower().split() # 사용자 입력 대기.
            if len(user_input) != 3: # 입력 3개 아닐 시 형식 오류.
                print("잘못된 입력 형식. 'action row col' 형식으로 입력.")
                continue # 입력 재요청.

            action = user_input[0] # 첫 번째 요소: 명령어 (s 또는 f).
            row = int(user_input[1]) # 두 번째 요소: 행 번호.
            col = int(user_input[2]) # 세 번째 요소: 열 번호.

            # 입력 좌표 보드 범위 밖 여부 확인.
            if not (0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE):
                print(f"좌표 범위 초과. 0부터 {BOARD_SIZE - 1} 사이 값 입력.")
                continue # 입력 재요청.

            if action == 's': # 's' 명령 시 삽질.
                shovel(row, col)
            elif action == 'f': # 'f' 명령 시 깃발 꽂기/뽑기.
                flag(row, col)
            else: # 's' 또는 'f' 아닌 경우.
                print("잘못된 명령어. 's'나 'f' 사용.")
        except ValueError: # 숫자 아닌 입력 시.
            print("잘못된 입력. 행과 열은 정수 입력 필수.")
        except Exception as e: # 예상치 못한 다른 에러 발생 시.
            print(f"예상치 못한 오류 발생: {e}")

    display_board() # 게임 종료 후 최종 보드 상태 출력.
    print("게임 종료. Thanks For Playing.")

# 게임 시작을 위한 `play_minesweeper()` 함수 호출.
play_minesweeper()

Welcome to Text-Based Minesweeper!
명령어는 'action row col' 형태로 입력.
명령어 종류: 's'는 삽으로 파헤치기, 'f'는 깃발 꽂기/뽑기.
예시: 's 0 0'은 (0, 0) 칸 파헤치기, 'f 1 2'는 (1, 2) 칸에 깃발 꽂기/뽑기.


  0 1 2 3 4 5 6 7 8
  ------------------
0| # # # # # # # # #
1| # # # # # # # # #
2| # # # # # # # # #
3| # # # # # # # # #
4| # # # # # # # # #
5| # # # # # # # # #
6| # # # # # # # # #
7| # # # # # # # # #
8| # # # # # # # # #
Flags remaining: 10


# 텍스트 기반 지뢰찾기 게임

이것은 Python으로 구현된 간단한 텍스트 기반 지뢰찾기 게임입니다. 사용자는 콘솔을 통해 게임 보드와 상호작용하며 지뢰를 피하고 모든 안전한 칸을 여는 것을 목표로 합니다.

## 게임 규칙

- 보드는 9x9 격자로 구성됩니다.
- 총 10개의 지뢰가 무작위로 배치됩니다.
- 숫자가 표시된 칸은 주변 8칸에 있는 지뢰의 개수를 나타냅니다.
- 0이 표시된 빈 칸을 열면 주변의 다른 빈 칸과 숫자가 있는 칸이 자동으로 열립니다 (BFS).

## 플레이 방법

게임이 시작되면 보드가 표시됩니다. 명령어는 `action row col` 형태로 입력합니다.

- **칸 파헤치기 (열기)**: `s row col`
  - 예시: `s 0 0` (0행 0열 칸을 엽니다.)
  - 지뢰가 없는 칸을 열면 숫자가 표시되거나 주변 칸이 자동으로 열립니다.
  - 지뢰가 있는 칸을 열면 게임이 종료됩니다.

- **깃발 꽂기/뽑기**: `f row col`
  - 예시: `f 1 2` (1행 2열 칸에 깃발을 꽂거나 뽑습니다.)
  - 깃발은 지뢰가 있다고 생각하는 칸에 표시하여 실수로 여는 것을 방지합니다.
  - 깃발은 닫힌 칸에만 꽂거나 뽑을 수 있습니다.

## 승리 조건

모든 지뢰가 없는 칸을 성공적으로 열면 게임에서 승리합니다.

## 게임 오버 조건

지뢰가 있는 칸을 파헤치면 게임 오버가 됩니다.

## 코드 구조

주요 함수와 변수는 다음과 같습니다:

- `BOARD_SIZE`, `NUM_MINES`: 게임 보드의 크기(9x9)와 총 지뢰 개수(10개)를 설정하는 상수입니다. 게임의 난이도와 규모를 조절할 수 있습니다.

- `array`: `BOARD_SIZE` x `BOARD_SIZE` 크기의 2차원 리스트로, 각 칸의 정보를 저장합니다. 지뢰가 있는 칸은 100, 지뢰가 없는 칸은 주변 8칸에 있는 지뢰의 개수(0~8)를 나타냅니다.

- `mine`: `BOARD_SIZE` x `BOARD_SIZE` 크기의 2차원 리스트로, 실제 지뢰의 위치를 표시합니다. 지뢰가 있으면 1, 없으면 0입니다. 이 배열은 지뢰 탐지 시에 사용됩니다.

- `not_mine`: `BOARD_SIZE` x `BOARD_SIZE` 크기의 2차원 리스트로, '지뢰가 아닌 칸'을 추적하는 데 사용됩니다. 초기에는 모든 칸이 1(닫힘)로 설정되며, 지뢰가 배치되면 해당 칸은 `None`으로 변경됩니다. 승리 조건 확인에 사용됩니다.

- `flag_qty`: 현재 남은 깃발의 개수를 나타내는 정수 변수입니다. 지뢰 배치 시 `NUM_MINES`로 초기화되며, 깃발을 꽂거나 뽑을 때마다 값이 변경됩니다.

- `game`: 게임의 현재 진행 상태를 나타내는 불리언(boolean) 변수입니다. `True`이면 게임이 진행 중이고, `False`이면 게임이 종료된 상태입니다.

- `time_elapsed`: 게임 시작 후 경과 시간을 기록하는 변수입니다. 현재 텍스트 기반 게임에서는 사용되지 않습니다.

- `opened_arr`: `BOARD_SIZE` x `BOARD_SIZE` 크기의 2차원 리스트로, 각 칸의 열림 상태를 기록합니다. 칸이 열리면 1, 닫혀 있으면 `None`입니다.

- `flagged_arr`: `BOARD_SIZE` x `BOARD_SIZE` 크기의 2차원 리스트로, 각 칸의 깃발 표시 상태를 기록합니다. 깃발이 꽂혀 있으면 1, 없으면 `None`입니다.

- `reset()`: 게임을 새로 시작할 때 호출되는 함수입니다. `array`, `mine`, `not_mine`, `opened_arr`, `flagged_arr` 등 모든 게임 관련 변수를 초기 상태로 재설정하고, `NUM_MINES`만큼의 지뢰를 보드에 무작위로 배치하며 주변 칸의 숫자를 업데이트합니다.

- `display_board()`: 현재 게임 보드의 상태를 콘솔에 출력하는 함수입니다. 열린 칸, 닫힌 칸, 깃발, 지뢰(`*` 표시) 등을 시각적으로 보여줍니다. 남은 깃발 개수도 함께 표시합니다.

- `game_set(is_game_running)`: 게임의 진행 상태(`game` 변수)를 설정하는 함수입니다. 게임 시작 시 `True`로, 게임 오버 또는 승리 시 `False`로 설정됩니다.

- `flag(x, y)`: 지정된 (x, y) 위치에 깃발을 꽂거나 이미 꽂혀 있는 깃발을 제거하는 함수입니다. 닫힌 칸에만 작동하며, `flag_qty`를 업데이트합니다.

- `click(x, y)`: 지정된 (x, y) 위치의 칸을 열린 상태로 변경하는 보조 함수입니다. `shovel` 함수 내부에서 호출됩니다.

- `Queue` 클래스: BFS(너비 우선 탐색) 구현을 위해 `collections.deque`를 사용하여 큐 자료구조를 제공합니다. `enqueue`, `dequeue`, `is_empty` 메서드를 포함합니다.

- `dx`, `dy`: BFS 탐색 시 현재 칸의 주변 8칸으로 이동하기 위한 x, y 좌표 변화량을 정의한 리스트입니다.

- `bfs(x, y)`: (x, y) 위치의 칸이 0(주변 지뢰 없음)일 경우 호출되는 너비 우선 탐색 함수입니다. 해당 칸과 연결된 모든 0인 칸 및 그 경계에 있는 숫자 칸들을 자동으로 엽니다.

- `check_win_condition()`: 게임의 승리 조건을 확인하는 함수입니다. 모든 지뢰가 아닌 칸이 개방되었는지 여부를 검사하여 `True` 또는 `False`를 반환합니다.

- `shovel(x, y)`: 사용자가 (x, y) 위치의 칸을 파헤칠 때 호출되는 주요 게임 로직 함수입니다. 선택된 칸이 지뢰이면 게임 오버를 처리하고, 지뢰가 아니면 해당 칸을 개방합니다. 개방된 칸이 0이면 `bfs`를 호출하여 주변 칸을 자동으로 열고, 승리 조건을 확인합니다.

- `play_minesweeper()`: 텍스트 기반 지뢰찾기 게임의 전체 흐름을 제어하는 메인 함수입니다. 게임을 초기화하고, 사용자 입력을 받아 `shovel` 또는 `flag` 함수를 호출하며 게임이 종료될 때까지 반복됩니다.